### Gradient Descent

In this notebook we will demonstrate how gradient descent works. We will use an example similar to the one from the slides: the price of drinks in a bar. We know that **1 drink costs 6 euros** and **2 drinks cost 12 euros** and we want a neuron to learn this relation, so that we can ask it how much 3 drinks would cost.

First, we will implement gradient descent ourselves with numpy, so that there is no magic involved. Then we will let Keras do the same job.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'    # has to be set before keras is imported

import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 6]

# Training data
x = np.array([1, 2])     # Number of drinks
y = np.array([6, 12])    # Price in euros

plt.plot(x, y, 'o'), plt.grid(True)
plt.xlabel('number of drinks'), plt.ylabel('price [EUR]')
plt.xlim(0, 3), plt.ylim(0, 14);

#### Definition of Neuron

Our neuron is the simplest one we can imagine, it only multiplies the input with its weight: $y = w x$

* **Training phase**: $x$ and $y$ are given, we look for the weight $w$
* **Inference phase**: $x$ and $w$ are given, we compute $y$

We start with an (intentionally wrong) weight of 4, as in the slides.

In [ ]:
class Neuron:
    def __init__(self, weight=4):
        self.weight = weight

    def __call__(self, x):
        return self.weight * x

    def __repr__(self):
        return f'Neuron weight: {self.weight:.5f}'

#### Definition of Model

We also need to define the neural network (or model) itself. In our simple case, the model consists of one single neuron.

In order to conduct the optimization, we need to define the **loss function** that tells us how wrong the model is. We will use the sum of squared errors

$$E = \sum (y_{pred} - y_{gt})^2 = \sum (w x - y_{gt})^2$$

and its derivative with respect to **all** of the network parameters (the so called **gradient**). Our network has only one parameter, so the gradient is a single number

$$\frac{dE}{dw} = \sum 2 (w x - y_{gt})\, x$$

The gradient tells us in which direction (and how fast) the loss grows when we increase the weight.

In [ ]:
class Network():
    def __init__(self):
        self.neuron = Neuron()

    def loss(self, x, y):
        return np.sum((self.neuron(x) - y)**2)

    def gradient(self, x, y):
        return np.sum(2 * (self.neuron(x) - y) * x)

#### Training

We can now apply the gradient descent algorithm to find the weight that minimizes the loss. In each iteration we make a step against the gradient (downhill), scaled by the **learning rate** `lr`:

$$w \leftarrow w - lr \cdot \frac{dE}{dw}$$

Note that in every line below, the weight, the gradient and the loss all belong to the *same* moment, i.e. before the weight is updated.

In [ ]:
model = Network()
lr = 0.05

weights, losses = [], []
for i in range(20):
    loss = model.loss(x, y)
    grad = model.gradient(x, y)
    weights.append(model.neuron.weight)
    losses.append(loss)
    print(f'{i:2d} | {model.neuron} | gradient {grad:9.4f} | loss {loss:.6f}')

    model.neuron.weight = model.neuron.weight - lr * grad

The weight converges to 6 euros per drink, which is exactly what we expected. Notice that the gradient (and with it the step size) shrinks automatically as we approach the minimum. In this example, every step halves the distance to the optimal weight (4 → 5 → 5.5 → 5.75 → ...).

Let's visualize what has happened. The loss $E$ as a function of the weight is a parabola and gradient descent simply slides down along it.

In [ ]:
w = np.linspace(0, 9, 100)
E = [np.sum((wi * x - y)**2) for wi in w]

plt.subplot(1, 2, 1)
plt.plot(w, E, 'k'), plt.plot(weights, losses, 'ro-'), plt.grid(True)
plt.xlabel('weight'), plt.ylabel('loss')
plt.title('Loss as a function of the weight')

plt.subplot(1, 2, 2)
plt.plot(losses, '.-'), plt.grid(True)
plt.xlabel('iteration'), plt.ylabel('loss')
plt.title('Loss during training');

#### Learning Rate

The learning rate is one of the most important parameters of the training. Let's try several values, always starting from the same weight.

In [ ]:
for lr in [0.05, 0.2, 0.25]:
    model = Network()
    weights = [model.neuron.weight]
    for _ in range(10):
        model.neuron.weight = model.neuron.weight - lr * model.gradient(x, y)
        weights.append(model.neuron.weight)
    plt.plot(weights, 'o-', label=f'lr = {lr}')

plt.axhline(6, color='k', linestyle='--'), plt.grid(True), plt.legend()
plt.xlabel('iteration'), plt.ylabel('weight')
plt.ylim(-10, 20);

* `lr = 0.05`: smooth convergence to the optimal weight
* `lr = 0.2`: the steps are too large, the weight jumps back and forth between 4 and 8 and never converges
* `lr = 0.25`: even larger steps, every jump overshoots more than the previous one and the training **diverges**

For this particular problem, the learning rate has to be smaller than 0.2 to converge. In real networks, there is no such simple rule and the learning rate has to be found by experimenting.

### Keras Implementation

Let's now implement the same neural network model with Keras. Keras is a high level API that takes care of the gradients, parameter updates and the training loop for us. Since version 3, Keras can run on top of different backends (TensorFlow, JAX, PyTorch). We have selected **PyTorch** in the first cell (via the `KERAS_BACKEND` variable), the framework we will also use later in this course.

In [ ]:
import keras
from keras import Model
from keras.layers import Dense, Input

We define the network with one input, and one neuron. Our hand-made neuron only multiplies the input with its weight, so we switch the bias off (`use_bias=False`) and, to be able to compare the results, we also start from the same initial weight 4. By default, Keras would initialize the weight randomly.

In [ ]:
inputs = Input(shape=(1,))
outputs = Dense(1, activation="linear", use_bias=False,
                kernel_initializer=keras.initializers.Constant(4))(inputs)
model = Model(inputs, outputs)

model.summary()

The summary confirms that the network has one single trainable parameter.

#### Model Training

Before the training, we have to *compile* the model, which means choosing the optimizer and the loss. We use plain stochastic gradient descent (`SGD`) and mean squared error.

Keras computes the **mean** (not the sum) of squared errors. With two training samples this makes the gradient 2x smaller, so we have to use a 2x larger learning rate (0.1) to reproduce the training from above (`lr = 0.05`).

In [ ]:
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1), loss='mean_squared_error')

One **epoch** is one pass through the entire training dataset. With `batch_size=2` the whole dataset (2 samples) is one batch, so every epoch makes exactly one weight update, which is exactly the gradient descent we have implemented. Let's train epoch by epoch and print the weight after each one.

In [ ]:
for epoch in range(8):
    model.fit(x, y, epochs=1, batch_size=2, verbose=0)
    print(f'epoch {epoch}: weight {model.layers[1].weights[0].numpy()[0, 0]:.5f}')

The weights are identical to the ones we got with our own implementation (5.0, 5.5, 5.75, ...), so Keras is doing nothing else than gradient descent.

#### Stochastic Gradient Descent

For real datasets, we cannot afford to compute the gradient over all the samples in every step (see the slides for the memory requirements). Instead, we split the data into **mini-batches** and update the weights after each batch. Here, `batch_size=1` means two updates per epoch, one for each sample. Let's restart from the initial weight and see how the training behaves.

In [ ]:
model.layers[1].set_weights([np.array([[4.]])])

for epoch in range(8):
    model.fit(x, y, epochs=1, batch_size=1, verbose=0)
    print(f'epoch {epoch}: weight {model.layers[1].weights[0].numpy()[0, 0]:.5f}')

Every epoch now makes two updates instead of one, so the weight moves faster per epoch (5.68 after the first epoch instead of 5.0). In general, a mini-batch gradient is only a noisy estimate of the full gradient, but it is much cheaper to compute. In this toy problem there is no noise, because both samples agree on the optimal weight.

Let's now run the training for longer and see where we end up. After the training, we can check the parameter that the network has learnt and use the model for inference: how much do 3 drinks cost?

In [ ]:
model.fit(x, y, epochs=100, batch_size=1, verbose=0)

print('Learnt weight:', model.layers[1].weights[0].numpy()[0, 0])
print('Price of 3 drinks:', model.predict(np.array([3]), verbose=0)[0, 0])